# Session 30: Hyperparameter Tuning

Data Science with Python, Level 2, Stage 4: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Tell parameters from hyperparameters.
2. Search a grid with GridSearchCV.
3. Read cv_results_ and best_params_.
4. Search wide spaces with RandomizedSearchCV.
5. Tune pipeline steps with step__setting names.
6. Keep a final test set for an honest score.

## Where We Are

A model learns its **parameters** from data: the coefficients of a regression, the split points of a tree. But we choose its **hyperparameters** before training: the regularisation strength `C`, the tree depth, the learning rate, the number of trees.

So far we picked them by hand or tried a few in a loop. Today you will search for them systematically with scikit-learn's search tools, which combine every setting with cross-validation. You will also see an important truth: tuning helps, but usually less than good data and good features.

## Setup

The setup cell builds the churn pipeline from Session 29 and splits off a 20 percent test set that we only use at the very end.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     train_test_split, cross_val_score)
from sklearn.metrics import roc_auc_score

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=0)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                          OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
pipe = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))])

## 1. Parameters and Hyperparameters

| Model | Learned parameters | Hyperparameters we choose |
| --- | --- | --- |
| Linear / logistic regression | Coefficients, intercept | C (regularisation), penalty |
| Decision tree | Split features and thresholds | max_depth, min_samples_leaf |
| Random forest | All the trees | n_estimators, max_features, min_samples_leaf |
| Gradient boosting | All the trees | learning_rate, max_iter, max_depth |
| K-Means | Cluster centres | n_clusters |

## 2. GridSearchCV

In [ ]:
grid = {"model__C": [0.001, 0.01, 0.1, 1, 10, 100]}
search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc")
search.fit(X_dev, y_dev)

print("Best setting:", search.best_params_)
print("Best CV AUC:", round(search.best_score_, 4))

In [ ]:
results = pd.DataFrame(search.cv_results_)
print(results[["param_model__C", "mean_test_score", "std_test_score",
               "rank_test_score"]].round(4))

In [ ]:
C = results["param_model__C"].astype(float)
plt.errorbar(C, results["mean_test_score"],
             yerr=results["std_test_score"], marker="o", capsize=4)
plt.xscale("log")
plt.xlabel("C (log scale)")
plt.ylabel("Mean CV AUC")
plt.title("Strong regularisation hurts, then flat")
plt.show()

In [ ]:
grid = {
    "model__C": [0.01, 0.1, 1],
    "model__class_weight": [None, "balanced"],
    "prep__num__simpleimputer__strategy": ["mean", "median"],
}
search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc").fit(X_dev, y_dev)
print(len(search.cv_results_["params"]), "combinations")
for name, value in search.best_params_.items():
    print(name, "=", value)

## 3. RandomizedSearchCV

In [ ]:
pipe.set_params(model=HistGradientBoostingClassifier(random_state=0))
default_auc = cross_val_score(pipe, X_dev, y_dev, cv=5, scoring="roc_auc").mean()
print("Default boosting CV AUC:", round(default_auc, 4))

In [ ]:
space = {
    "model__learning_rate": loguniform(0.01, 0.3),
    "model__max_depth": [2, 3, 4, 6, None],
    "model__max_iter": randint(50, 400),
    "model__min_samples_leaf": randint(10, 100),
    "model__l2_regularization": uniform(0, 1),
}
search = RandomizedSearchCV(pipe, space, n_iter=30, cv=5, scoring="roc_auc",
                            random_state=0, n_jobs=-1)
search.fit(X_dev, y_dev)
print("Best CV AUC:", round(search.best_score_, 4))

In [ ]:
best = search.best_params_
for name, value in best.items():
    if isinstance(value, float):
        value = round(float(value), 3)
    print(name.replace("model__", ""), value)

## 4. An Honest Final Score

In [ ]:
logistic = GridSearchCV(
    Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=1000))]),
    {"model__C": [0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc").fit(X_dev, y_dev)

for name, s in [("logistic", logistic), ("boosting", search)]:
    auc = roc_auc_score(y_test, s.predict_proba(X_test)[:, 1])
    print(f"{name:9} CV {s.best_score_:.3f}  test {auc:.3f}")

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Tuning on test | Picking settings by test set scores | Search with CV on the development data only. |
| Huge grids | A grid with thousands of combinations | Use RandomizedSearchCV with a fixed n_iter. |
| Linear ranges | C in [1, 2, 3, 4, 5] | Use powers of ten: 0.01, 0.1, 1, 10. |
| Chasing decimals | Celebrating +0.001 AUC | Compare with the fold standard deviation. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Tune a Tree

Use GridSearchCV to tune `max_depth` (2 to 10) and `min_samples_leaf` (1, 10, 50) of a `DecisionTreeClassifier` inside the churn pipeline. Print the best settings and score.

In [ ]:
# Write your code below

### In-Class Exercise 2: Heatmap of a Grid

From the tree search, make a table of mean CV AUC with depth as rows and leaf size as columns (`pivot_table` on `cv_results_`) and show it with `seaborn.heatmap`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Forest Random Search

Run a RandomizedSearchCV (20 combinations) for a random forest in the churn pipeline over `n_estimators` (100 to 500), `max_depth` ([4, 8, 12, None]), `min_samples_leaf` (1 to 50) and `max_features` (["sqrt", 0.5, 1.0]).

In [ ]:
# Write your code below

### In-Class Exercise 4: Tune for Recall

Search `model__C` and `model__class_weight` for the logistic pipeline with `scoring="recall"` instead of AUC. How do the best settings change, and why?

In [ ]:
# Write your code below

## Summary

- Parameters are learned; hyperparameters are chosen.
- GridSearchCV tries every combination with cross-validation.
- RandomizedSearchCV samples a fixed number from wide ranges.
- Pipeline settings use `step__setting` names.
- `best_params_`, `best_score_` and `cv_results_` explain the search.
- Test the chosen model once; small gains may be noise.

## What's Next

Tuning gave small gains. Better features often give more. Next session you will create features from dates, ratios and groups, and explain any model's predictions with permutation importance and partial dependence plots.